In [37]:
import pandas as pd
import joblib

x_train_scaled, x_test_scaled, y_train, y_test = joblib.load('../data/processed/train_test_data.pkl')

In [38]:
x_train_scaled.shape, x_test_scaled.shape, y_train.shape, y_test.shape

((5634, 23), (1409, 23), (5634,), (1409,))

In [39]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(random_state=42)
baseline_model.fit(x_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solve

In [40]:
y_pred_baseline = baseline_model.predict(x_test_scaled)

In [41]:
from sklearn.metrics import accuracy_score

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

Baseline Accuracy: 0.8070


## Baseline Model — Logistic Regression

A simple Logistic Regression model was trained as our baseline.
It achieved 80.70% accuracy, notably higher than the majority-class
baseline of ~73.5%, confirming the model is learning real patterns.
However, accuracy alone doesn't tell the full story given class imbalance —
precision, recall, and F1-score will be evaluated properly once we compare
multiple models.

## Model 2: Decision Tree

A Decision Tree makes predictions by asking a series of yes/no questions
about the features (e.g., "Is Contract = Month-to-month? Is tenure < 12?"),
splitting the data at each step until it reaches a final prediction.

**Why try this model:** EDA showed clear rule-like patterns (e.g., low tenure
+ month-to-month contract → high churn), which Decision Trees are naturally
good at capturing.

**Advantages:** highly interpretable, captures non-linear relationships and
feature interactions automatically, no scaling required.

**Disadvantages:** prone to overfitting — without limits, it can grow deep
enough to memorize the training data instead of learning general patterns,
making it unstable and less reliable on unseen data.

**Key hyperparameter:** `max_depth` limits how many questions deep the tree
can go, controlling

In [42]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(x_train_scaled, y_train)

y_pred_dt = dt_model.predict(x_test_scaled)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
print(f"Decision Tree Accuracy: {dt_accuracy:.4f}")

Decision Tree Accuracy: 0.7942


**Result:** Decision Tree achieved 79.42% accuracy — slightly lower than
the Logistic Regression baseline (80.70%). A single tree with limited
depth may be too simple to capture the full pattern complexity; ensemble
methods (Random Forest, Gradient Boosting) may perform better.

Model 3: Random Forest

How it works (beginner level): instead of building one Decision Tree, Random Forest builds many trees (often 100+), where each tree is trained on a random subset of the data and a random subset of features. When predicting, all trees "vote," and the majority vote becomes the final prediction. This is called an ensemble method — combining many "weak" or imperfect individual models into one stronger collective model.

Why it might work here: it directly addresses our Decision Tree's weakness — a single tree can overfit or make mistakes based on the particular splits it happened to choose, but averaging across 100+ trees, each seeing slightly different random subsets, smooths out those individual errors and captures more robust patterns.

In [43]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(x_train_scaled, y_train)

y_pred_rf = rf_model.predict(x_test_scaled)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

Random Forest Accuracy: 0.8055


**Result:** Random Forest achieved 80.55% accuracy — comparable to, but
slightly below, the Logistic Regression baseline. This suggests the
underlying relationships in this dataset may be fairly linear, since a
simpler model performs competitively against a more complex ensemble.

## Model 4: Gradient Boosting

Gradient Boosting builds trees sequentially, where each new tree focuses
on correcting the errors of previous trees, rather than building
independent trees like Random Forest.

**Why try this model:** this sequential error-correction approach often
achieves higher accuracy than Random Forest on tabular data.

**Advantages:** often more accurate than Random Forest, provides feature
importance.

**Disadvantages:** slower to train (sequential, not parallel), more
sensitive to hyperparameter choices, less interpretable than a single tree.

**Key hyperparameters:** `n_estimators` (number of trees), `learning_rate`
(how aggressively each tree corrects previous errors).

In [44]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(x_train_scaled, y_train)

y_pred_gb = gb_model.predict(x_test_scaled)

gb_accuracy = accuracy_score(y_test, y_pred_gb)
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")

Gradient Boosting Accuracy: 0.7977


**Result:** Gradient Boosting achieved 79.77% accuracy — also slightly
below the Logistic Regression baseline, continuing the pattern seen with
Decision Tree and Random Forest. Tree-based models aren't outperforming
the simpler linear model on this dataset.

## Model 5: K-Nearest Neighbors (KNN)

KNN predicts a new customer's churn by finding the "K" most similar
customers (based on feature distance) in the training data and taking
a majority vote among them.

**Why try this model:** it's fundamentally different from the tree-based
approaches tried so far, relying on similarity rather than explicit rules
— a useful, diverse comparison point.

**Advantages:** simple, intuitive concept; no real training phase.

**Disadvantages:** slow at prediction time on large datasets; very
sensitive to feature scaling (this is exactly why scaling in Phase 5 was
essential); no feature importance or easy interpretability.

**Key hyperparameter:** `n_neighbors` — how many nearby neighbors to
consider when voting.

In [45]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train_scaled, y_train)

y_pred_knn = knn_model.predict(x_test_scaled)

knn_accuracy = accuracy_score(y_test, y_pred_knn)
print(f"KNN Accuracy: {knn_accuracy:.4f}")

KNN Accuracy: 0.7473


In [46]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': (baseline_model, y_pred_baseline),
    'Decision Tree': (dt_model, y_pred_dt),
    'Random Forest': (rf_model, y_pred_rf),
    'Gradient Boosting': (gb_model, y_pred_gb),
    'KNN': (knn_model, y_pred_knn)
}

results = []

for name, (model, y_pred) in models.items():
    y_proba = model.predict_proba(x_test_scaled)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results)
results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.806955,0.659375,0.564171,0.608069,0.841923
1,Decision Tree,0.794180,0.631250,0.540107,0.582133,0.826704
2,Random Forest,0.805536,0.674825,0.516043,0.584848,0.841773
3,Gradient Boosting,0.797729,0.653979,0.505348,0.570136,0.841404
4,KNN,0.747339,0.525281,0.500000,0.512329,0.771742


## Model Comparison Summary

All 5 models were evaluated on Accuracy, Precision, Recall, F1-Score, and
ROC-AUC. **Logistic Regression** achieved the best F1-Score (0.6081) and
tied for the best ROC-AUC (0.8419), making it the strongest overall
candidate despite being the simplest model tried.

Recall was moderate across all models (0.50-0.56), meaning even the best
model misses close to half of actual churners — a real limitation that
could be addressed with techniques like class-weighting in future work.

KNN performed clearly worst across every metric and will not be considered
further. Logistic Regression and Random Forest (second-best F1/ROC-AUC)
will be carried forward to hyperparameter tuning in Phase 10.

In [47]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [48]:
from sklearn.model_selection import GridSearchCV

log_reg_params = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear']
}

log_reg_grid = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid=log_reg_params,
    scoring='f1',
    cv=5
)

log_reg_grid.fit(x_train_scaled, y_train)

print("Best parameters:", log_reg_grid.best_params_)
print("Best F1 score (cross-validated):", log_reg_grid.best_score_)

Best parameters: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Best F1 score (cross-validated): 0.598955954532479


## Hyperparameter Tuning: Logistic Regression

Used GridSearchCV with 5-fold cross-validation, optimizing for F1-score.
Tested combinations of `C` (regularization strength) and confirmed `L2`
regularization performs best.

**Best parameters:** `C=10, penalty='l2', solver='liblinear'`
**Best cross-validated F1-score:** 0.599 — consistent with the untuned
model's test F1 (0.608), confirming the default settings were already
close to optimal for this dataset.

In [49]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_params,
    scoring='f1',
    cv=5
)

rf_grid.fit(x_train_scaled, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best F1 score (cross-validated):", rf_grid.best_score_)

Best parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
Best F1 score (cross-validated): 0.5640952935147064


## Hyperparameter Tuning: Random Forest

Used GridSearchCV with 5-fold cross-validation across `n_estimators`,
`max_depth`, and `min_samples_split`.

**Best parameters:** `n_estimators=200, max_depth=10, min_samples_split=5`
**Best cross-validated F1-score:** 0.564 — lower than tuned Logistic
Regression (0.599), confirming Logistic Regression as the stronger model
for this dataset even after tuning.

## Tuning Conclusion

Logistic Regression, even in its original untuned form, remained
competitive with or better than every other model tried — including after
dedicated hyperparameter tuning of both itself and Random Forest. This
suggests the churn patterns in this dataset are largely linear and
well-captured by a simple, interpretable model. **Logistic Regression
(tuned: C=10, penalty='l2') is selected as our final model going into
Phase 11 (Feature Importance).**

In [52]:
feature_names = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
                  'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                  'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                  'PaperlessBilling', 'MonthlyCharges', 'TotalCharges',
                  'InternetService_Fiber optic', 'InternetService_No',
                  'Contract_One year', 'Contract_Two year',
                  'PaymentMethod_Credit card (automatic)',
                  'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

best_log_reg = log_reg_grid.best_estimator_

coefficients = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_log_reg.coef_[0]
})

coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values('Abs_Coefficient', ascending=False)
coefficients

,Feature,Coefficient,Abs_Coefficient
14,MonthlyCharges,-2.151482,2.151482
4,tenure,-1.299210,1.299210
16,InternetService_Fiber optic,1.278678,1.278678
17,InternetService_No,-1.068015,1.068015
19,Contract_Two year,-0.586934,0.586934
15,TotalCharges,0.577155,0.577155
12,StreamingMovies,0.454252,0.454252
11,StreamingTV,0.453442,0.453442
6,MultipleLines,0.316673,0.316673
18,Contract_One year,-0.283602,0.283602


## Feature Importance (Logistic Regression Coefficients)

**Top predictors:**
- `MonthlyCharges` (strongest, negative direction *after* controlling for
  InternetService) and `tenure` (negative — matches EDA exactly) are the
  two most influential features.
- `InternetService_Fiber optic` (positive) is a strong, non-obvious churn
  driver — fiber optic customers churn more, possibly due to reliability
  issues, pricing, or competitive alternatives.
- `Contract_Two year` and `Contract_One year` (both negative) confirm our
  EDA finding: longer contracts strongly reduce churn risk.

**Note:** MonthlyCharges' negative coefficient initially appears to
contradict EDA (which showed churners had higher median charges) — this
is resolved by understanding that coefficients reflect each feature's
effect *after controlling for* other features (like InternetService)
simultaneously, which EDA alone cannot show.

**Weakest predictors:** `gender`, `Partner`, `PaymentMethod_Credit card
(automatic)`, and `TechSupport` had negligible influence on the model's
predictions.